In [1]:
import pandas as pd                  # Pandas
import numpy as np                   # Numpy
from matplotlib import pyplot as plt # Matplotlib

# Package to implement ML Algorithms
import sklearn
from sklearn.tree import DecisionTreeClassifier # DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier # RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier # AdaBoostClassifier
from sklearn.ensemble import VotingClassifier # VotingClassifier
from sklearn.inspection import permutation_importance

from sklearn.metrics import f1_score

# Package to visualize Decision Tree
from sklearn import tree

# Package for generating confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Package for generating classification report
from sklearn.metrics import classification_report

# Package for data partitioning
from sklearn.model_selection import train_test_split

# Package to record time
import time

# Module to save and load Python objects to and from files
import pickle 

# Ignore Deprecation Warnings
import warnings
warnings.filterwarnings('ignore')

# Display inline plots as vector-based (svg)
%config InlineBackend.figure_formats = ['svg']

%matplotlib inline

In [2]:
df = pd.read_csv('Application_Data.csv')
df = df.drop(columns = ['Applicant_ID'])
df['Status'] = df['Status'].replace({
    1: "Approved",
    0: "Rejected"
})
df = df.replace(r'^\s+|\s+$', '', regex=True)
df.head()

,Applicant_Gender,Owned_Car,Owned_Realty,Total_Children,Total_Income,Income_Type,Education_Type,Family_Status,Housing_Type,Owned_Mobile_Phone,Owned_Work_Phone,Owned_Phone,Owned_Email,Job_Title,Total_Family_Members,Applicant_Age,Years_of_Working,Total_Bad_Debt,Total_Good_Debt,Status
0,M,1,1,0,112500,Working,Secondary / secondary special,Married,House / apartment,1,0,0,0,Security staff,2,59,4,0,30,Approved
1,F,0,1,0,270000,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1,53,9,0,5,Approved
2,F,0,1,0,270000,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1,53,9,0,5,Approved
3,F,0,1,0,270000,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1,53,9,0,27,Approved
4,F,0,1,0,270000,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1,53,9,0,39,Approved


In [3]:
# Distribution of Species column
df['Status'].value_counts(normalize = True)

Status
Approved    0.995185
Rejected    0.004815
Name: proportion, dtype: float64

In [4]:
df.dtypes

Applicant_Gender        object
Owned_Car                int64
Owned_Realty             int64
Total_Children           int64
Total_Income             int64
Income_Type             object
Education_Type          object
Family_Status           object
Housing_Type            object
Owned_Mobile_Phone       int64
Owned_Work_Phone         int64
Owned_Phone              int64
Owned_Email              int64
Job_Title               object
Total_Family_Members     int64
Applicant_Age            int64
Years_of_Working         int64
Total_Bad_Debt           int64
Total_Good_Debt          int64
Status                  object
dtype: object

In [5]:
# Select input and output features
features = df.drop(columns = ['Status', 'Owned_Mobile_Phone'])
output = df['Status']

In [6]:
# One hot encoding for categorical variables
# Identifying categorical variables
cat_var = ["Applicant_Gender", "Owned_Car", "Owned_Realty", "Income_Type", "Education_Type", "Family_Status", "Housing_Type", "Owned_Work_Phone", "Owned_Phone", "Owned_Email", "Job_Title"]

# One-hot encoding of categorical variables
features_encoded = pd.get_dummies(features, columns = cat_var)

features_encoded.head()

,Total_Children,Total_Income,Total_Family_Members,Applicant_Age,Years_of_Working,Total_Bad_Debt,Total_Good_Debt,Applicant_Gender_F,Applicant_Gender_M,Owned_Car_0,...,Job_Title_Laborers,Job_Title_Low-skill Laborers,Job_Title_Managers,Job_Title_Medicine staff,Job_Title_Private service staff,Job_Title_Realty agents,Job_Title_Sales staff,Job_Title_Secretaries,Job_Title_Security staff,Job_Title_Waiters/barmen staff
0,0,112500,2,59,4,0,30,False,True,False,...,False,False,False,False,False,False,False,False,True,False
1,0,270000,1,53,9,0,5,True,False,True,...,False,False,False,False,False,False,True,False,False,False
2,0,270000,1,53,9,0,5,True,False,True,...,False,False,False,False,False,False,True,False,False,False
3,0,270000,1,53,9,0,27,True,False,True,...,False,False,False,False,False,False,True,False,False,False
4,0,270000,1,53,9,0,39,True,False,True,...,False,False,False,False,False,False,True,False,False,False


In [7]:
train_X, test_X, train_y, test_y = train_test_split(features_encoded, output, test_size = 0.2, random_state = 42)

In [8]:
# Random Forest
rf_clf = RandomForestClassifier(random_state = 42)

start = time.time()            # Start Time
rf_clf.fit(train_X, train_y)  
stop = time.time()             # End Time
print(f"Training time: {stop - start}s")

Training time: 1.0350923538208008s


In [9]:
# Random Forest Confusion Matrix
# Predictions on test set
y_pred = rf_clf.predict(test_X)

# Now generate confusion matrix
rf_cm = confusion_matrix(test_y, y_pred, labels = rf_clf.classes_)
rf_disp = ConfusionMatrixDisplay(confusion_matrix = rf_cm, display_labels = rf_clf.classes_)

# Specify figure size
fig, ax = plt.subplots(figsize = (5, 5))
plt.rcParams.update({'font.size': 12})

# Display Confusion Matrix
rf_disp.plot(cmap = 'Oranges', ax = ax)

# Save as SVG
plt.savefig("rf_confusion_matrix.svg", bbox_inches = 'tight');

In [10]:
# Random Forest Classification Report
rf_report = classification_report(test_y, y_pred, output_dict = True)
rf_report_df = pd.DataFrame(rf_report).T
rf_report_df

# Save the report as a CSV File
rf_report_df.to_csv('rf_class_report.csv') 

In [11]:
# Random Forest Feature Importance Plot
# Storing importance values from the trained model
rf_importance = rf_clf.feature_importances_

# Storing feature importance as a dataframe
rf_feature_imp = pd.DataFrame(list(zip(train_X.columns, rf_importance)),
               columns = ['Feature', 'Importance'])

rf_feature_imp = rf_feature_imp.sort_values('Importance', ascending = False).reset_index(drop = True)

# Bar plot
plt.figure(figsize = (10, 5))
plt.barh(rf_feature_imp['Feature'], rf_feature_imp['Importance'], color = ['orange', 'red'])

plt.xlabel("Importance")
plt.ylabel("Input Feature")
plt.yticks(fontsize = 5)
plt.title('Which features are the most important for credit card approval prediction?') 
plt.tight_layout()
plt.savefig("rf_feature_imp.svg");

In [12]:
# Pickle file: saving the trained RF model
# Creating the file where we want to write the model
rf_pickle = open('randomforest.pickle', 'wb') 

# Write DT model to the file
pickle.dump(rf_clf, rf_pickle) 

# Close the file
rf_pickle.close() 